# 02. Modelagem não supervisionada das denúncias do Disque 100

A unidade de análise é a **denúncia única (`hash`)**. O notebook consome a base agregada criada no Notebook 01, impedindo que denúncias com múltiplas linhas ou violações tenham peso artificialmente maior.


## 1. Configuração e carga


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score,davies_bouldin_score,calinski_harabasz_score,adjusted_rand_score

warnings.filterwarnings("ignore")
RANDOM_STATE=42
DATA_FILE=Path("resultados/disque100_denuncias_modelagem.parquet")
VIOL_FILE=Path("resultados/matriz_violacoes_por_denuncia.parquet")
OUTPUT_DIR=Path("resultados/modelagem")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

df=pd.read_parquet(DATA_FILE)
assert df["hash"].is_unique,"A base de modelagem deve ter um registro por hash."
print(df.shape); df.head()


## 2. Features derivadas


In [ ]:
df["data_de_cadastro"]=pd.to_datetime(df["data_de_cadastro"],errors="coerce")
df["ano_cadastro"]=df["data_de_cadastro"].dt.year
df["mes_cadastro"]=df["data_de_cadastro"].dt.month
df["trimestre_cadastro"]=df["data_de_cadastro"].dt.quarter
df["dia_semana_cadastro"]=df["data_de_cadastro"].dt.dayofweek
df["fim_de_semana"]=(df["dia_semana_cadastro"]>=5).astype("int8")

MAPA_REGIAO={"AC":"NORTE","AP":"NORTE","AM":"NORTE","PA":"NORTE","RO":"NORTE","RR":"NORTE","TO":"NORTE","AL":"NORDESTE","BA":"NORDESTE","CE":"NORDESTE","MA":"NORDESTE","PB":"NORDESTE","PE":"NORDESTE","PI":"NORDESTE","RN":"NORDESTE","SE":"NORDESTE","DF":"CENTRO-OESTE","GO":"CENTRO-OESTE","MT":"CENTRO-OESTE","MS":"CENTRO-OESTE","ES":"SUDESTE","MG":"SUDESTE","RJ":"SUDESTE","SP":"SUDESTE","PR":"SUL","RS":"SUL","SC":"SUL"}
for origem,destino in [("uf","regiao_ocorrencia"),("uf_da_vitima","regiao_vitima"),("uf_do_suspeito","regiao_suspeito")]:
    if origem in df: df[destino]=df[origem].map(MAPA_REGIAO)


## 3. Seleção das variáveis

**Excluídas:** identificador, data bruta, localidades de alta cardinalidade, campos textuais específicos e variáveis removidas por alta ausência no Notebook 01.

**Incluídas:** contexto da denúncia, perfis resumidos da vítima e do suspeito, tempo, região, quantidade de vítimas e indicadores binários de violações.


In [ ]:
EXCLUIR={"hash","data_de_cadastro","municipio","municipio_da_vitima","sl_suspeito_municipio","profissao_da_vitima","profissao_do_suspeito","motivacao","vitima_especificacao_etnia","suspeito_especificacao_etnia"}
CANDIDATAS=[
"canal_de_atendimento","denuncia_emergencial","denunciante","cenario_da_violacao","pais","uf","frequencia","inicio_das_violacoes","sl_quantidade_vitimas","grupo_vulneravel","relacao_vitima_suspeito","genero_da_vitima","orientacao_sexual_da_vitima","faixa_etaria_da_vitima","nacionalidade_da_vitima","deficiencia_da_vitima","doenca_rara_da_vitima","vitima_preso_a","pais_da_vitima","uf_da_vitima","grau_de_instrucao_da_vitima","raca_cor_da_vitima","natureza_juridica_do_suspeito","genero_do_suspeito","orientacao_sexual_do_suspeito","faixa_etaria_do_suspeito","suspeito_nacionalidade","deficiencia_do_suspeito","doenca_rara_do_suspeito","suspeito_preso","pais_do_suspeito","uf_do_suspeito","grau_de_instrucao_do_suspeito","raca_cor_do_suspeito","sl_suspeito_ramo","ano_cadastro","mes_cadastro","trimestre_cadastro","dia_semana_cadastro","fim_de_semana","regiao_ocorrencia","regiao_vitima","regiao_suspeito","qtd_tipos_violacao","multiplas_violacoes"]
VIOL_COLS=[c for c in df.columns if c.startswith("violacao__")]
FEATURES=[c for c in CANDIDATAS+VIOL_COLS if c in df.columns and c not in EXCLUIR]
X=df[FEATURES].copy()
num_cols=X.select_dtypes(include=["number","bool"]).columns.tolist()
cat_cols=[c for c in X.columns if c not in num_cols]
print({"features":len(FEATURES),"numericas":len(num_cols),"categoricas":len(cat_cols),"violacoes":len(VIOL_COLS)})


## 4. Pré-processamento


In [ ]:
numeric_pipe=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",RobustScaler())])
categorical_pipe=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",min_frequency=0.01,sparse_output=True))])
preprocessor=ColumnTransformer([("num",numeric_pipe,num_cols),("cat",categorical_pipe,cat_cols)],sparse_threshold=1.0,verbose_feature_names_out=False)
X_pre=preprocessor.fit_transform(X)
feature_names=preprocessor.get_feature_names_out()
selector=VarianceThreshold(0.0)
X_sel=selector.fit_transform(X_pre)
selected_names=feature_names[selector.get_support()]
print("Matriz processada:",X_sel.shape)


## 5. Redução de dimensionalidade

Como o one-hot gera matriz esparsa, usa-se `TruncatedSVD`. Os componentes são usados na clusterização para reduzir ruído e custo computacional.


In [ ]:
max_components=min(50,X_sel.shape[1]-1,X_sel.shape[0]-1)
svd=TruncatedSVD(n_components=max_components,random_state=RANDOM_STATE)
X_svd=svd.fit_transform(X_sel)
acum=np.cumsum(svd.explained_variance_ratio_)
n80=int(np.searchsorted(acum,0.80)+1)
n_model=max(2,min(n80,max_components))
X_model=X_svd[:,:n_model]
print({"componentes_disponiveis":max_components,"componentes_80pct":n80,"usados":n_model,"variancia_acumulada":round(acum[n_model-1],3)})
plt.plot(range(1,len(acum)+1),acum); plt.axhline(.80,color="red",ls="--"); plt.xlabel("Componentes"); plt.ylabel("Variância acumulada"); plt.show()


## 6. Clusterização K-Means


In [ ]:
resultados=[]
for k in range(2,11):
    model=KMeans(n_clusters=k,n_init=30,random_state=RANDOM_STATE)
    labels=model.fit_predict(X_model)
    resultados.append({"algoritmo":"KMeans","k":k,"silhouette":silhouette_score(X_model,labels),"davies_bouldin":davies_bouldin_score(X_model,labels),"calinski_harabasz":calinski_harabasz_score(X_model,labels),"inercia":model.inertia_})
aval_k=pd.DataFrame(resultados)
display(aval_k)
BEST_K=int(aval_k.sort_values(["silhouette","davies_bouldin"],ascending=[False,True]).iloc[0]["k"])
kmeans=KMeans(n_clusters=BEST_K,n_init=50,random_state=RANDOM_STATE)
df["cluster_kmeans"]=kmeans.fit_predict(X_model)
print("BEST_K:",BEST_K); display(df["cluster_kmeans"].value_counts().sort_index())


## 7. Clusterização hierárquica e baseada em densidade


In [ ]:
hier=AgglomerativeClustering(n_clusters=BEST_K,linkage="ward")
df["cluster_hierarquico"]=hier.fit_predict(X_model)

dbscan=DBSCAN(eps=1.0,min_samples=max(5,int(np.sqrt(len(df)))))
df["cluster_dbscan"]=dbscan.fit_predict(X_model)
print(df["cluster_dbscan"].value_counts().sort_index())


## 8. Avaliação interna e estabilidade


In [ ]:
avaliacoes=[]
for nome,col in [("KMeans","cluster_kmeans"),("Hierarquico-Ward","cluster_hierarquico")]:
    lab=df[col].to_numpy()
    avaliacoes.append({"algoritmo":nome,"silhouette":silhouette_score(X_model,lab),"davies_bouldin":davies_bouldin_score(X_model,lab),"calinski_harabasz":calinski_harabasz_score(X_model,lab)})
avaliacoes=pd.DataFrame(avaliacoes)
display(avaliacoes)

ref=df["cluster_kmeans"].to_numpy(); estabilidade=[]
for seed in range(20):
    lab=KMeans(n_clusters=BEST_K,n_init=20,random_state=seed).fit_predict(X_model)
    estabilidade.append({"seed":seed,"adjusted_rand":adjusted_rand_score(ref,lab)})
estabilidade=pd.DataFrame(estabilidade)
display(estabilidade.describe())


## 9. Associação entre violações


In [ ]:
matriz=pd.read_parquet(VIOL_FILE).set_index("hash").astype(bool)
# requer: pip install mlxtend
try:
    from mlxtend.frequent_patterns import apriori, association_rules
    itemsets=apriori(matriz,min_support=0.01,use_colnames=True,max_len=4)
    regras=association_rules(itemsets,metric="lift",min_threshold=1.0)
    regras=regras.query("confidence >= 0.20").sort_values(["lift","confidence","support"],ascending=False)
    display(regras[["antecedents","consequents","support","confidence","lift"]].head(50))
except ImportError:
    regras=pd.DataFrame()
    print("Instale mlxtend para executar Apriori: pip install mlxtend")


## 10. Detecção de anomalias


In [ ]:
iforest=IsolationForest(n_estimators=500,contamination="auto",random_state=RANDOM_STATE,n_jobs=-1)
df["anomalia_iforest"]=iforest.fit_predict(X_model)
df["score_anomalia_iforest"]=iforest.decision_function(X_model)
display(df.nsmallest(30,"score_anomalia_iforest")[["hash","score_anomalia_iforest","cluster_kmeans","qtd_tipos_violacao","sl_quantidade_vitimas"]])


## 11. Interpretação dos clusters


In [ ]:
# Tamanho
tamanho=df.groupby("cluster_kmeans").size().rename("n").to_frame()
tamanho["pct"]=(tamanho["n"]/len(df)*100).round(2)
display(tamanho)

# Numéricas nas escalas originais
num_interpret=[c for c in ["sl_quantidade_vitimas","qtd_tipos_violacao","ano_cadastro","mes_cadastro"] if c in df]
display(df.groupby("cluster_kmeans")[num_interpret].median())

# Categoria modal e distribuição relativa
def moda(s):
    m=s.dropna().mode(); return m.iloc[0] if not m.empty else np.nan
cat_interpret=[c for c in ["cenario_da_violacao","frequencia","grupo_vulneravel","genero_da_vitima","faixa_etaria_da_vitima","raca_cor_da_vitima","genero_do_suspeito","faixa_etaria_do_suspeito","relacao_vitima_suspeito","regiao_ocorrencia"] if c in df]
display(df.groupby("cluster_kmeans")[cat_interpret].agg(moda))

# Violações mais características por cluster
viol_cols=[c for c in df.columns if c.startswith("violacao__")]
perfil_viol=df.groupby("cluster_kmeans")[viol_cols].mean().T
for cluster in perfil_viol.columns:
    display(perfil_viol[cluster].sort_values(ascending=False).head(15).to_frame(f"cluster_{cluster}"))


## 12. Visualização bidimensional


In [ ]:
pca2=PCA(n_components=2,random_state=RANDOM_STATE)
coords=pca2.fit_transform(X_model)
plot_df=pd.DataFrame({"Componente 1":coords[:,0],"Componente 2":coords[:,1],"cluster":df["cluster_kmeans"].astype(str)})
plt.figure(figsize=(10,7)); sns.scatterplot(data=plot_df,x="Componente 1",y="Componente 2",hue="cluster",alpha=.6,s=30); plt.title("Projeção dos clusters"); plt.tight_layout(); plt.show()


## 13. Exportação


In [ ]:
df.to_parquet(OUTPUT_DIR/"denuncias_com_clusters_e_anomalias.parquet",index=False)
avaliacoes.to_csv(OUTPUT_DIR/"avaliacao_modelos.csv",index=False,encoding="utf-8-sig")
estabilidade.to_csv(OUTPUT_DIR/"estabilidade_kmeans.csv",index=False,encoding="utf-8-sig")
tamanho.to_csv(OUTPUT_DIR/"tamanho_clusters.csv",encoding="utf-8-sig")
if not regras.empty: regras.to_csv(OUTPUT_DIR/"regras_associacao.csv",index=False,encoding="utf-8-sig")
print("Resultados em",OUTPUT_DIR.resolve())


## 14. Conclusões e cautelas

Registre os perfis encontrados, a estabilidade, as regras substantivamente relevantes e as anomalias que merecem inspeção. Evite interpretar clusters como categorias reais ou causais. Considere subnotificação, ausência não aleatória e mudanças administrativas no período.
